# 01 — Chain

Spin up Anvil, deploy `BandwidthEscrow` + `BandwidthNFT`, walk one trade, decode events.

Prereqs: `anvil` and `forge` on PATH.

## Setup

In [ ]:
import sys, pathlib
_ROOT = pathlib.Path.cwd().resolve()
if (_ROOT / 'shared').is_dir():
    sys.path.insert(0, str(_ROOT))
elif (_ROOT.parent / 'shared').is_dir():
    sys.path.insert(0, str(_ROOT.parent))


In [ ]:
from shared.anvil import anvil
from shared.config import Config
from shared.deploy import deploy_contracts
from shared.chain import make_web3, send_tx, STATUS_NAMES
from shared.contracts import get_escrow_contract, get_nft_contract
from eth_account import Account

DEPLOYER = '0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80'
PROVIDER = '0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d'
CONSUMER = '0x5de4111afa1a4b94908f83103eb1f1706367c2e68ca870fc3fb9a804cdab365a'
provider_account = Account.from_key(PROVIDER)
consumer_account = Account.from_key(CONSUMER)

## Build (start Anvil + deploy)

In [ ]:
ctx = anvil(port=18545)
rpc_url = ctx.__enter__()
print('anvil:', rpc_url)

cfg = Config(rpc_url=rpc_url, deployer_private_key=DEPLOYER,
             provider_private_key=PROVIDER,
             consumer_private_key=CONSUMER, sdn_mock=True)
addrs = deploy_contracts(cfg)
print('escrow:', addrs['bandwidthEscrow'])
print('nft:   ', addrs['bandwidthNFT'])

## Run (one trade end-to-end)

In [ ]:
w3 = make_web3(cfg)
escrow = get_escrow_contract(w3)
nft = get_nft_contract(w3)

agreement_id = 1234
mbps, duration, price_wei = 5, 600, 10**16  # 0.01 ETH

# 1. Consumer locks payment
tx, _ = send_tx(w3, consumer_account, CONSUMER,
                escrow.functions.requestAgreement(
                    agreement_id, provider_account.address,
                    mbps, duration),
                value=price_wei)
print('requestAgreement tx:', tx)

# 2. Provider mints NFT bound to (agreement, mbps, duration)
tx, mint_receipt = send_tx(w3, provider_account, PROVIDER,
    nft.functions.mint(provider_account.address, agreement_id,
                       mbps, duration, 'clab://pe1/eth-1.100'))
from shared.chain import extract_token_id
token_id = extract_token_id(mint_receipt, nft)
print('minted tokenId:', token_id)

# 3. Provider approves escrow then deposits — atomic swap
send_tx(w3, provider_account, PROVIDER,
        nft.functions.approve(escrow.address, token_id))
send_tx(w3, provider_account, PROVIDER,
        escrow.functions.deposit(agreement_id, token_id))
print('swap complete')

## Inspect

In [ ]:
ag = escrow.functions.getAgreement(agreement_id).call()
print('status:', STATUS_NAMES[ag[7]])
print('owner of NFT:', nft.functions.ownerOf(token_id).call())
print('consumer was:', consumer_account.address)

## Teardown

In [ ]:
ctx.__exit__(None, None, None)